In [ ]:
from pbixray import PBIXRay

model = PBIXRay(r"C:\Users\Blog Demo - June.pbix")

In [ ]:
tables = model.tables
print(tables)

In [ ]:
dax_tables = model.dax_tables
print(dax_tables)

In [ ]:
import zipfile
import json
import os
from io import BytesIO

def extract_pbix_components(pbix_path, output_dir="pbix_blog"):
    components = {}
    os.makedirs(output_dir, exist_ok=True)

    try:
        with zipfile.ZipFile(pbix_path, 'r') as pbix_zip:
            print(f"Extracting from: {pbix_path}")

            for file_name in pbix_zip.namelist():
                with pbix_zip.open(file_name) as f:
                    content = f.read()
                    components[file_name] = content

                    safe_path = os.path.join(output_dir, file_name.replace('/', '_'))
                    with open(safe_path, "wb") as out_file:
                        out_file.write(content)

            print(f"Extracted {len(components)} components")

            if 'Report/Layout' in components:
                layout_json = json.loads(components['Report/Layout'])
                layout_path = os.path.join(output_dir, 'layout.json')
                with open(layout_path, 'w', encoding='utf-8') as f:
                    json.dump(layout_json, f, indent=2)
                print(f"Report Layout saved as {layout_path}")

            if 'Metadata' in components:
                metadata_path = os.path.join(output_dir, 'Metadata.txt')
                with open(metadata_path, 'wb') as f:
                    f.write(components['Metadata'])
                print(f"Metadata saved as {metadata_path}")

            if 'DataModel' in components:
                datamodel_path = os.path.join(output_dir, 'DataModel.data')
                with open(datamodel_path, 'wb') as f:
                    f.write(components['DataModel'])
                print(f"DataModel saved (binary) at {datamodel_path}")

            for name in components:
                if name.lower().endswith(('.png', '.jpg', '.jpeg')):
                    img_path = os.path.join(output_dir, os.path.basename(name))
                    with open(img_path, 'wb') as f:
                        f.write(components[name])
                    print(f"Image saved: {img_path}")

    except Exception as e:
        print(f"Error extracting PBIX components: {e}")

    return components

# Example usage
pbix_file_path = r"C:\Users\Blog Demo - June.pbix"
extract_pbix_components(pbix_file_path)


In [ ]:
import os
import json
import struct
import pandas as pd
import io
import zlib

def attempt_read_data_file(file_path):
    """Try different methods to read a Power BI .data file"""
    results = {}
    
    # Method 1: Try basic binary reading
    try:
        with open(file_path, 'rb') as f:
            binary_data = f.read()
            results['file_size'] = len(binary_data)
            
            # Check for compression header
            is_compressed = False
            if binary_data[:2] == b'\x78\x9c' or binary_data[:2] == b'\x78\xda':
                is_compressed = True
                results['compression'] = 'zlib detected'
            
            # Display the first 100 bytes as hex for inspection
            results['hex_preview'] = binary_data[:100].hex(' ')
    except Exception as e:
        results['binary_read_error'] = str(e)
    
    # Method 2: Try decompression if it appears to be zlib compressed
    if is_compressed:
        try:
            decompressed = zlib.decompress(binary_data)
            results['decompressed_size'] = len(decompressed)
            results['decompressed_preview'] = decompressed[:100].hex(' ')
        except Exception as e:
            results['decompression_error'] = str(e)
    
    # Method 3: Try to parse as a structured binary format
    try:
        with open(file_path, 'rb') as f:
            # Read the first 8 bytes as a potential header
            header = f.read(8)
            results['potential_header'] = header.hex(' ')
            
            # Try to interpret as various binary formats
            f.seek(0)
            # Try to read as a series of 4-byte floats
            float_data = []
            while True:
                bytes_data = f.read(4)
                if not bytes_data or len(bytes_data) < 4:
                    break
                try:
                    value = struct.unpack('f', bytes_data)[0]
                    float_data.append(value)
                except:
                    pass
            
            if float_data:
                results['float_sample'] = float_data[:10]
    except Exception as e:
        results['structured_read_error'] = str(e)
    
    return results

def extract_data_with_json_context(data_file_path, json_file_path):
    """Use the JSON schema to interpret the .data file"""
    
    # First read the JSON file to understand the structure
    try:
        with open(json_file_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        # Extract table schemas if present
        tables = []
        if isinstance(metadata, dict):
            # Look for common schema patterns in Power BI JSON
            if 'model' in metadata and 'tables' in metadata['model']:
                tables = metadata['model']['tables']
            elif 'tables' in metadata:
                tables = metadata['tables']
        
        table_schemas = []
        for table in tables:
            if isinstance(table, dict) and 'columns' in table:
                table_schemas.append({
                    'name': table.get('name', 'Unknown'),
                    'columns': [{
                        'name': col.get('name', 'Unknown'),
                        'dataType': col.get('dataType', 'Unknown')
                    } for col in table['columns']]
                })
        
        # Now attempt to read the data file with this schema information
        data_results = attempt_read_data_file(data_file_path)
        
        # Return combined results
        return {
            'schema': table_schemas,
            'data_analysis': data_results
        }
    except Exception as e:
        return {
            'error': str(e),
            'data_analysis': attempt_read_data_file(data_file_path)
        }

def try_all_extraction_methods(data_file_path, json_file_path=None):
    """Try multiple methods to extract data from a Power BI .data file"""
    
    results = {}
    
    # Method 1: Basic .data file analysis
    results['basic_analysis'] = attempt_read_data_file(data_file_path)
    
    # Method 2: If JSON file is available, use it for context
    if json_file_path and os.path.exists(json_file_path):
        results['json_guided_extraction'] = extract_data_with_json_context(data_file_path, json_file_path)
    
    # Method 3: Try parsing as a structured data format (CSV, parquet, etc.)
    try:
        # Try as CSV
        df = pd.read_csv(data_file_path, error_bad_lines=False, nrows=5)
        results['csv_sample'] = df.to_dict()
    except:
        results['csv_error'] = "Could not parse as CSV"
    
    try:
        # Try as parquet
        df = pd.read_parquet(data_file_path)
        results['parquet_sample'] = df.head(5).to_dict()
    except:
        results['parquet_error'] = "Could not parse as Parquet"
    
    return results

def main():
    # Replace these with your actual file paths
    data_file_path = r"C:\DB\pbix\pbix_blog\DataModel.data"
    json_file_path = r"C:\DB\pbix\pbix_blog\Report_Custom.json"
    
    print("Analyzing Power BI .data file...")
    results = try_all_extraction_methods(data_file_path, json_file_path)
    
    # Save the results to a JSON file for inspection
    with open("extraction_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    
    print("Analysis complete. Results saved to extraction_results.json")
    
    # If we successfully parsed any data, save it to a CSV
    if 'csv_sample' in results and results['csv_sample']:
        pd.DataFrame(results['csv_sample']).to_csv("extracted_data.csv", index=False)
        print("Sample data saved to extracted_data.csv")
    elif 'parquet_sample' in results and results['parquet_sample']:
        pd.DataFrame(results['parquet_sample']).to_csv("extracted_data.csv", index=False)
        print("Sample data saved to extracted_data.csv")

if __name__ == "__main__":
    main()

Analyzing Power BI .data file...
Analysis complete. Results saved to extraction_results.json
